# 8c — M4 single-dataset standalone (median-beat, wavelet_env): train **ningbo**, cross-test **ptbxl**

Mirror of M3 07b/07c. Demonstrates the **batch effect** (AP collapses cross-dataset while AUC holds) that justifies the **combined** M4 (08a). NOT a deployed detector. Reuses the deployed-winner features `m4_features_wavelet_env.csv` filtered by source. Single-dataset gate (no cross criterion); selection = max OOF AP, depth open, parsimony tiebreak. XGBoost only, NaN-native, Platt. **fold10 + cross fold9/10 UNTOUCHED.**


### Block 8c.0: Setup & single-dataset split

In [ ]:
# M4 single-dataset detector (median-beat, wavelet_env): train=ningbo | cross-test=ptbxl.
# PURPOSE: demonstrate the batch effect (AP collapses cross-dataset while AUC holds = detection transfers,
# score scale does not) -> justifies the COMBINED M4 (08a) + percentile display. NOT a deployed detector.
# Reuses the deployed-winner features m4_features_wavelet_env.csv, filtered by source. Single-dataset gate
# (|d|>0.3 & p_FDR & IC95, NO cross-dataset criterion); dedup>0.95; selection = max OOF AP depth-open +
# parsimony tiebreak (M3/M4 rule). XGBoost only, NaN-native, Platt. fold10 + cross fold9/10 UNTOUCHED.
import os,sys,json,warnings,time
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
warnings.filterwarnings('ignore'); EPS=1e-9
TRAIN_DS='ningbo'; CROSS_DS='ptbxl'; DET='wavelet_env'   # the A/B-winning detector deployed in 08a
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); MODELS=os.path.join(ROOT,'models',f'M4_medianbeat_{TRAIN_DS}')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics'); SRC=os.path.join(ROOT,'src')
os.makedirs(MODELS,exist_ok=True); sys.path.insert(0,SRC)
from evaluation import evaluate_standard
TAG=TRAIN_DS; TIE_EPS=0.01
FEATURES_CSV=os.path.join(PROCESSED,f'm4_features_{DET}.csv')
_hdr=pd.read_csv(FEATURES_CSV,nrows=0).columns.tolist()
_sc={'ecg_id','patient_id','source'}; _ic={'label','fold','extraction_failed','n_beats'}
df=pd.read_csv(FEATURES_CSV,dtype={c:(str if c in _sc else 'int16' if c in _ic else 'float32') for c in _hdr})
METAC=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
ALL_FEATS=[c for c in df.columns if c not in METAC]
dtr=df[df.source==TRAIN_DS].reset_index(drop=True); dcr=df[df.source==CROSS_DS].reset_index(drop=True)
tr=dtr[dtr.fold.between(1,8)]
print(f"M4 single-dataset (median-beat, {DET}): train={TRAIN_DS} | cross-test={CROSS_DS}")
print(f"{TRAIN_DS}: {len(dtr)} ECGs, {int((dtr.label==1).sum())} WPW | WPW/fold {dtr[dtr.label==1].fold.value_counts().sort_index().to_dict()}")
print(f"{CROSS_DS} (cross): {len(dcr)} ECGs, {int((dcr.label==1).sum())} WPW")
print(f"Available features: {len(ALL_FEATS)} (med/ext/var)")
print(f"Train(1-8) WPW: {int((tr.label==1).sum())} | Val(9): {int((dtr[dtr.fold==9].label==1).sum())} | Test(10): {int((dtr[dtr.fold==10].label==1).sum())} [UNTOUCHED]")
assert int((tr.label==1).sum())>20, "too few train WPW - features CSV incomplete? (copy full m4_features_wavelet_env.csv from backups/M4_v1/processed/)"
print("Block setup OK.")


### Block 8c.0b: Patient-leakage check (within train dataset)

In [ ]:
# Patient-leakage check within the TRAIN dataset (blocking assert).
pat=dtr.groupby('patient_id')['fold'].nunique(); leaky=pat[pat>1]
print(f"{TRAIN_DS}: {dtr.patient_id.nunique()} patients, {len(dtr)} ECGs | in >1 fold: {len(leaky)}")
assert len(leaky)==0, f"PATIENT LEAK: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage.")


### Block 8c.1: Single-dataset gate (|d|>0.3 & p_FDR<0.05 & IC95) + dedup>0.95

In [ ]:
# Helpers + single-dataset gate (NO cross-dataset criterion; this notebook MEASURES cross-transfer, not gate it).
def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))
def ap_boot(yy,pp,n=200,seed=42):
    rng=np.random.default_rng(seed); pos,neg=np.where(yy==1)[0],np.where(yy==0)[0]; a=np.empty(n)
    for i in range(n):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]); a[i]=average_precision_score(yy[idx],pp[idx])
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)
def make_xgb(spw,**kw):
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,
           min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=6)
    p.update(kw); return XGBClassifier(**p)
def _view(c): return ('median' if c.startswith('med_') else 'extreme' if c.startswith('ext_') else 'variability' if c.startswith('var_') else '?')
SEL=os.path.join(PROCESSED,f'm4_{TAG}_selection_{DET}.csv')
if os.path.exists(SEL) and set(pd.read_csv(SEL,usecols=['feature']).feature)==set(ALL_FEATS):
    res=pd.read_csv(SEL); print(f"[guard] gate reloaded ({len(res)})")
else:
    w,n=tr[tr.label==1],tr[tr.label==0]; rows=[]
    for c in tqdm(ALL_FEATS,desc=f'Gate ({TRAIN_DS})',unit='feat'):
        a,b=w[c].values.astype(float),n[c].values.astype(float); d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b); ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'p_raw':p,'ci_excl0':ci_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna(); res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True); res.to_csv(SEL,index=False)
passed=res[res.gate].feature.tolist(); print(f"Pass the gate: {len(passed)}")
def dedup_fast(feats,thr=0.95):
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0); ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
dedup_list=dedup_fast(passed) if passed else []
print(f"After dedup>0.95: {len(dedup_list)} features (= k_max) | by view {pd.Series(dedup_list).map(_view).value_counts().to_dict()}")
print(res[res.gate].head(15)[['feature','d','p_FDR']].to_string(index=False))


### Block 8c.2: AP vs k — same (OOF +CI) vs cross (the batch effect)

In [ ]:
# Modeling matrices + AP-vs-k SAME (OOF folds 1-8, +95% CI) vs CROSS (other dataset, folds 1-8).
# The whole point of 08b/08c: watch AP_cross collapse while AUC_cross holds -> batch effect.
y=tr.label.values; folds=tr.fold.values
FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
f9=dtr[dtr.fold==9].reset_index(drop=True); y9=f9.label.values
dcr8=dcr[dcr.fold.between(1,8)].reset_index(drop=True); ycr=dcr8.label.values   # cross-test = CROSS_DS folds 1-8 (its 9/10 stay vierge)
Xtr=tr[dedup_list].to_numpy(np.float32); X9=f9[dedup_list].to_numpy(np.float32); Xcr=dcr8[dedup_list].to_numpy(np.float32)
spw=(y==0).sum()/max((y==1).sum(),1)
print(f"matrices: train {Xtr.shape} | val9 {X9.shape} ({int(y9.sum())} WPW) | cross {Xcr.shape} ({int(ycr.sum())} WPW)")
def oof_xgb(k,**kw):
    X=Xtr[:,:k]; oof=np.full(len(tr),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof
def oof_ap(k,**kw):
    oof=oof_xgb(k,**kw); m=~np.isnan(oof); return float(average_precision_score(y[m],oof[m]))
RK=os.path.join(PROCESSED,f'm4_{TAG}_ap_vs_k_{DET}.csv')
KMAX=len(dedup_list); KGRID=list(range(2,KMAX+1,2))
if os.path.exists(RK) and len(pd.read_csv(RK))>=len(KGRID)-1:
    rk=pd.read_csv(RK); print(f"[guard] AP-vs-k reloaded ({len(rk)} pts)")
else:
    rows=[]
    for k in tqdm(KGRID,desc='k (same+cross)',unit='k'):
        oof=oof_xgb(k); m=~np.isnan(oof); aps=float(average_precision_score(y[m],oof[m]))
        mdl=make_xgb(spw).fit(Xtr[:,:k],y); scr=mdl.predict_proba(Xcr[:,:k])[:,1]
        rows.append({'k':k,'AP_same':aps,'AP_cross':float(average_precision_score(ycr,scr)),'AUC_cross':float(roc_auc_score(ycr,scr))})
        if k%20==0: pd.DataFrame(rows).to_csv(RK,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK,index=False)
kbest=int(rk.loc[rk.AP_same.idxmax(),'k']); ap_same_max=float(rk.AP_same.max())
# plateau = smallest k within 1% of max
plateau=int(rk[rk.AP_same>=ap_same_max-0.01].k.min())
print(f"max AP_same {ap_same_max:.3f}@{kbest} | plateau(>=max-0.01) k={plateau} | k_max={KMAX}")
row_p=rk[rk.k==plateau].iloc[0]
print(f"At k={plateau}: AP_same {row_p.AP_same:.3f} | AP_cross {row_p.AP_cross:.3f} | AUC_cross {row_p.AUC_cross:.3f}  <- AP collapses, AUC holds = batch effect")
# bootstrap CI band on AP_same (coarse: every ~10th k)
oofp=oof_xgb(plateau); mp=~np.isnan(oofp); med,lo,hi=ap_boot(y[mp],oofp[mp])
print(f"AP_same @plateau bootstrap: {med:.3f} CI95[{lo:.3f},{hi:.3f}]")
CIB=os.path.join(PROCESSED,f'm4_{TAG}_ap_ci_{DET}.csv')
if os.path.exists(CIB): cib=pd.read_csv(CIB)
else:
    sub=rk.iloc[::max(1,len(rk)//14)]; rr=[]
    for k in tqdm(sub.k.tolist(),desc='CI band',unit='k'):
        oo=oof_xgb(int(k)); m=~np.isnan(oo); m2,l2,h2=ap_boot(y[m],oo[m]); rr.append({'k':int(k),'lo':l2,'hi':h2})
    cib=pd.DataFrame(rr); cib.to_csv(CIB,index=False)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(rk.k,rk.AP_same,lw=1.2,color='tab:blue',label=f'AP same OOF folds 1-8 ({TRAIN_DS})')
ax.fill_between(cib.k,cib.lo,cib.hi,alpha=0.15,color='tab:blue',label='95% bootstrap CI')
ax.plot(rk.k,rk.AP_cross,lw=1.2,color='tab:red',label=f'AP cross ({CROSS_DS}, folds 1-8)')
ax.plot(rk.k,rk.AUC_cross,lw=1.0,ls='--',color='tab:green',label=f'AUC cross ({CROSS_DS})')
ax.axhline(0.002 if CROSS_DS=='ningbo' else 0.003,ls=':',color='grey',lw=0.8,label='cross prevalence')
ax.set_xlabel('k'); ax.set_ylabel('AP / AUC'); ax.set_ylim(0,1.0)
ax.set_title(f'M4 {TRAIN_DS} (median-beat) - same vs cross: AP collapses, AUC holds [batch effect]')
ax.legend(fontsize=8,loc='center right'); plt.tight_layout()
P=os.path.join(FIG,f'M4_{TRAIN_DS}_samecross_apvsk.png'); plt.savefig(P,dpi=150,bbox_inches='tight'); plt.show(); print('-> figure:',P)


### Block 8c.3: Overfit diag + K×config grid + selection

In [ ]:
# Overfit diag (depth 2/3/4 at plateau) + K x config grid + selection (max OOF AP, depth open, parsimony tiebreak).
CFGS={'d2_lr05':dict(max_depth=2,learning_rate=0.05),'d2_lr10':dict(max_depth=2,learning_rate=0.10),
      'd3_lr05':dict(max_depth=3,learning_rate=0.05),'d3_lr10':dict(max_depth=3,learning_rate=0.10),
      'd4_lr05':dict(max_depth=4,learning_rate=0.05)}
def perfold_ap(k,**kw):
    oof=oof_xgb(k,**kw); out=[]
    for h in sorted(np.unique(folds)):
        m=(folds==h)&(~np.isnan(oof))
        if y[m].sum()>0: out.append(average_precision_score(y[m],oof[m]))
    return float(np.mean(out)),float(np.std(out)),float(np.min(out))
def ap_train(k,**kw):
    mdl=make_xgb(spw,**kw).fit(Xtr[:,:k],y); return float(average_precision_score(y,mdl.predict_proba(Xtr[:,:k])[:,1]))
print("diag @plateau k=%d (gap train-OOF, inter-fold variance):"%plateau)
for cn in ['d2_lr05','d3_lr05','d4_lr05']:
    kw=CFGS[cn]; a=oof_ap(plateau,**kw); t=ap_train(plateau,**kw); mu,sd,mn=perfold_ap(plateau,**kw)
    print(f"  {cn}: OOF {a:.3f} | train {t:.3f} | gap {t-a:+.3f} | per-fold {mu:.3f}+/-{sd:.3f} (min {mn:.3f})")
KS=sorted(set([20,40,80,plateau,kbest,min(KMAX,160),min(KMAX,220),KMAX]))
KS=[k for k in KS if 2<=k<=KMAX]
print(f"KS grid: {KS}")
GR=os.path.join(PROCESSED,f'm4_{TAG}_grid2d_{DET}.csv')
if os.path.exists(GR): grid=pd.read_csv(GR)
else:
    rows=[]
    for k in tqdm(KS,desc='K x config',unit='K'):
        for cn,kw in CFGS.items():
            a=oof_ap(k,**kw); t=ap_train(k,**kw); mu,sd,mn=perfold_ap(k,**kw)
            rows.append({'K':k,'cfg':cn,'depth':kw['max_depth'],'OOF_AP':a,'AP_train':t,'gap':t-a,'perfold_mean':mu,'perfold_std':sd,'perfold_min':mn})
    grid=pd.DataFrame(rows); grid.to_csv(GR,index=False)
print(grid.sort_values('OOF_AP',ascending=False).head(8)[['K','cfg','depth','OOF_AP','AP_train','gap','perfold_mean','perfold_min']].to_string(index=False))
# selection: max OOF AP (depth open); parsimony tiebreak within TIE_EPS -> fewest K then lowest depth
best=grid.OOF_AP.max(); cand=grid[grid.OOF_AP>=best-TIE_EPS].sort_values(['K','depth']).iloc[0]
K=int(cand.K); CFG=CFGS[cand.cfg]; CFGNAME=cand.cfg
print(f">>> SELECTED: K={K} {CFGNAME} depth{CFG['max_depth']} | OOF {cand.OOF_AP:.3f} | gap {cand.gap:+.3f} | per-fold {cand.perfold_mean:.3f}+/-{cand.perfold_std:.3f} (min {cand.perfold_min:.3f})")


### Block 8c.4: Standardized eval (same fold9 + cross) + freeze

In [ ]:
# Final fit at SELECTED (K,CFG): OOF (same) + Platt + multiseed + evaluate_standard SAME (fold9) and CROSS,
# then the same-vs-cross batch-effect summary, then FREEZE to models/M4_medianbeat_{TRAIN_DS}.
XtrK=Xtr[:,:K]; X9K=X9[:,:K]; XcrK=Xcr[:,:K]
oof_raw=np.full(len(tr),np.nan)
for trm,vam in FOLD_MASKS:
    if y[trm].sum()==0 or y[vam].sum()==0: continue
    oof_raw[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**CFG).fit(XtrK[trm],y[trm]).predict_proba(XtrK[vam])[:,1]
mask=~np.isnan(oof_raw); ap_oof=float(average_precision_score(y[mask],oof_raw[mask]))
platt=LogisticRegression(max_iter=1000).fit(oof_raw[mask].reshape(-1,1),y[mask])
xgb_raw=make_xgb(spw,**CFG).fit(XtrK,y)
ap_tr=float(average_precision_score(y,xgb_raw.predict_proba(XtrK)[:,1]))
score9=xgb_raw.predict_proba(X9K)[:,1]; score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
scorecr=xgb_raw.predict_proba(XcrK)[:,1]; scorecr_cal=platt.predict_proba(scorecr.reshape(-1,1))[:,1]
# multiseed fold9 (same)
seeds=[42,1,7,13,99]; aps9=[]
for s in seeds:
    m=make_xgb(spw,**CFG); m.set_params(random_state=s); m.fit(XtrK,y); aps9.append(average_precision_score(y9,m.predict_proba(X9K)[:,1]))
ms_mean,ms_std=float(np.mean(aps9)),float(np.std(aps9))
print(f"OOF AP (same, folds 1-8) {ap_oof:.3f} | gap train-OOF {ap_tr-ap_oof:+.3f} | multiseed fold9 {ms_mean:.3f}+/-{ms_std:.3f}")
# screening readout
def prec_at_rec(yy,pp,r=0.8):
    from sklearn.metrics import precision_recall_curve
    pr,rc,_=precision_recall_curve(yy,pp); ok=rc[:-1]>=r
    return float(pr[:-1][ok].max()) if ok.any() else 0.0
print(f"OOF prec@rec0.8 {prec_at_rec(y[mask],oof_raw[mask]):.3f}")
# evaluate_standard SAME (held-out fold9)
S=evaluate_standard(name=f'M4_{TRAIN_DS}',y_oof=y[mask],score_oof=oof_raw[mask],y_test=y9,score_test=score9,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_tr,
    multiseed={'AP_mean':ms_mean,'AP_std':ms_std,'seeds':seeds},
    extra={'mode':'single_dataset','train':TRAIN_DS,'detector':DET,'K':K,'cfg':CFGNAME,'ap_oof':ap_oof})
print(f"  SAME {TRAIN_DS}: AP {S['AP']:.3f} | AUC {S['AUC']:.3f}")
# evaluate_standard CROSS (other dataset, threshold from SAME OOF)
C=evaluate_standard(name=f'M4_{TRAIN_DS}_CROSS_on_{CROSS_DS}',y_oof=y[mask],score_oof=oof_raw[mask],y_test=ycr,score_test=scorecr,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=scorecr_cal,ap_train=ap_tr,
    extra={'mode':'cross_dataset','train':TRAIN_DS,'cross':CROSS_DS,'detector':DET,'K':K})
print(f"  CROSS on {CROSS_DS}: AP {C['AP']:.3f} | AUC {C['AUC']:.3f}")
# the batch-effect result
print("\n=== same vs cross (batch effect) ===")
print(f"  AP   same({TRAIN_DS}) {S['AP']:.3f}  ->  cross({CROSS_DS}) {C['AP']:.3f}   (collapse: {S['AP']-C['AP']:+.3f})")
print(f"  AUC  same({TRAIN_DS}) {S['AUC']:.3f}  ->  cross({CROSS_DS}) {C['AUC']:.3f}   (held: {S['AUC']-C['AUC']:+.3f})")
print(f"  => detection transfers (AUC), score scale does not (AP) -> single-dataset M4 is NOT externally valid; the COMBINED M4 (08a) is.")
# FREEZE
joblib.dump(xgb_raw,os.path.join(MODELS,f'm4_{TAG}_{DET}_model_raw.joblib'))
joblib.dump(platt,os.path.join(MODELS,f'm4_{TAG}_{DET}_platt.joblib'))
pd.DataFrame({'ecg_id':tr.ecg_id.values[mask],'proba_raw':oof_raw[mask],'label':y[mask]}).to_csv(os.path.join(PROCESSED,f'm4_{TAG}_{DET}_oof.csv'),index=False)
cfg_out={'model':'M4_medianbeat_single','train':TRAIN_DS,'cross':CROSS_DS,'detector':DET,'K':K,'cfg':CFGNAME,'depth':CFG['max_depth'],
         'features':dedup_list[:K],'ap_oof_same':ap_oof,'ap_train':ap_tr,'multiseed_fold9':{'mean':ms_mean,'std':ms_std},
         'same':{'AP':S['AP'],'AUC':S['AUC']},'cross':{'AP':C['AP'],'AUC':C['AUC']},'frozen':datetime.now().isoformat()}
json.dump(cfg_out,open(os.path.join(MODELS,f'm4_{TAG}_{DET}_config.json'),'w'),indent=2)
print(f"\n=== M4 {TRAIN_DS} FROZEN -> models/M4_medianbeat_{TRAIN_DS} (fold10 + cross fold9/10 untouched) ===")
